<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Lighter

Manages the Lighter data download and processing using the [Lighter REST API](https://apidocs.lighter.xyz/).
Candles (OHLCV) are downloaded from `https://mainnet.zklighter.elliot.ai/api/v1/candles` and saved,
one file per trading pair, into a folder named after the exchange (e.g. `../data/lighter`).

Notes:

- Lighter returns at most 500 candles per request, so full backfills require pagination.
- Candle timestamps (`t` field) are in milliseconds; API parameters (`start_timestamp`, `end_timestamp`) are in seconds.
- Spot markets are discovered via the explorer API at `https://explorer.elliot.ai/api/markets`.
- Symbols use `/` as separator (e.g. `LIT/USDC`).

** Finally, datetime columns are in UTC. **

In [0]:
#| echo: false
#| output: asis
show_doc(lighter_spot_markets)

---

### lighter_spot_markets

```python
def lighter_spot_markets(
    explorer_url:str='https://explorer.elliot.ai'
):
```

*Retrieve all spot markets from Lighter via the explorer API.*

Args:
    explorer_url (str): Lighter explorer API base URL. Defaults to mainnet explorer.

Returns:
    pandas.DataFrame: DataFrame with columns:
        - symbol: Trading pair symbol (e.g. 'LIT/USDC')
        - market_id: Integer market ID used for candle requests

In [0]:
#| echo: false
#| output: asis
show_doc(retry_fetch_candles)

---

### retry_fetch_candles

```python
def retry_fetch_candles(
    market_id, resolution, start_ts, end_ts, count_back:int=500, max_retries:int=3,
    base_url:str='https://mainnet.zklighter.elliot.ai', verbose:bool=False
):
```

*Fetch a single page of candles from Lighter, retrying on failure.*

Args:
    market_id (int): Lighter market ID
    resolution (str): Candle resolution (e.g. '1h', '1d')
    start_ts (int): Start timestamp in seconds since epoch
    end_ts (int): End timestamp in seconds since epoch
    count_back (int): Number of candles to fetch (max 500). Defaults to 500
    max_retries (int): Maximum retries before raising. Defaults to 3
    base_url (str): Lighter API base URL
    verbose (bool): If True, prints progress. Defaults to False

Returns:
    list: List of candle dicts with keys t (ms), o, h, l, c, v, V, i

In [0]:
#| echo: false
#| output: asis
show_doc(scrape_candles)

---

### scrape_candles

```python
def scrape_candles(
    market_id, timeframe, since, end:NoneType=None, max_retries:int=3, limit:int=500,
    base_url:str='https://mainnet.zklighter.elliot.ai', verbose:bool=False
):
```

*Download candles in pages of `limit` bars between `since` and `end`.*

Lighter's `count_back` parameter returns the most recent N candles before
`end_timestamp`, so we paginate **backwards**: after each fetch, we move
`end_timestamp` to just before the oldest candle received and repeat until
we reach `since` or the API returns no more data.

Args:
    market_id (int): Lighter market ID
    timeframe (str): Candle timeframe (e.g. '1m', '1h', '1d')
    since (int or str): Start time in seconds since epoch or ISO 8601 string
    end (int or str, optional): End time in seconds or ISO 8601. Defaults to now
    max_retries (int): Retries per page. Defaults to 3
    limit (int): Candles per request (max 500). Defaults to 500
    base_url (str): Lighter API base URL
    verbose (bool): If True, prints progress. Defaults to False

Returns:
    list: List of candle dicts with keys t (ms), o, h, l, c, v, V, i

In [0]:
#| echo: false
#| output: asis
show_doc(candles_to_df)

---

### candles_to_df

```python
def candles_to_df(
    candles, symbol
):
```

*Convert a list of Lighter candle dicts into a tidy DataFrame.*

Args:
    candles (list): List of candle dicts with keys t (ms), o, h, l, c, v
    symbol (str): Trading pair symbol added as the `pair` column

Returns:
    pandas.DataFrame: DataFrame with columns:
        - datetime: Candle timestamp (UTC, timezone-aware)
        - open, high, low, close, volume: OHLCV values
        - pair: Trading pair symbol
    Sorted by datetime with duplicate timestamps removed.

In [0]:
#| echo: false
#| output: asis
show_doc(lighter_candles)

---

### lighter_candles

```python
def lighter_candles(
    symbol:str='LIT/USDC', timeframe:str='1h', since:NoneType=None, end:NoneType=None, max_retries:int=3,
    limit:int=500, base_url:str='https://mainnet.zklighter.elliot.ai', explorer_url:str='https://explorer.elliot.ai',
    verbose:bool=False
):
```

*Download candles for a symbol from Lighter.*

Args:
    symbol (str, optional): Trading pair symbol (e.g. 'LIT/USDC'). Defaults to 'LIT/USDC'
    timeframe (str, optional): Candle timeframe. Defaults to '1h'
    since (int or str, optional): Start time (seconds since epoch or ISO 8601).
        If None, downloads the most recent ~`limit` candles.
    end (int or str, optional): End time (seconds or ISO 8601). Defaults to now
    max_retries (int, optional): Retries per page. Defaults to 3
    limit (int, optional): Candles per request (max 500). Defaults to 500
    base_url (str): Lighter API base URL
    explorer_url (str): Lighter explorer API base URL for market lookup
    verbose (bool, optional): If True, prints progress. Defaults to False

Returns:
    pandas.DataFrame: Tidy OHLCV DataFrame (see `candles_to_df`)

#### Example / tests

In [ ]:
#|eval: false
# Live test: spot markets are available
markets = lighter_spot_markets()
assert not markets.empty
assert 'symbol' in markets.columns
assert 'market_id' in markets.columns
assert 'LIT/USDC' in markets['symbol'].tolist()
markets.head()

,symbol,market_id
0,AAVE/USDC,2052
1,SKY/USDC,2053
2,AZTEC/USDC,2055
3,LIT/USDC,2049
4,UNI/USDC,2051


In [ ]:
#|eval: false
# Quick live test: download ~2 days of hourly LIT/USDC candles
start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ')
df_test = lighter_candles(symbol='LIT/USDC', timeframe='1h', since=start)
assert not df_test.empty
assert list(df_test.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
assert df_test['datetime'].dt.tz is not None
assert df_test['datetime'].is_monotonic_increasing
assert (df_test['pair'] == 'LIT/USDC').all()
assert len(df_test) > 24
df_test.tail()

,datetime,open,high,low,close,volume,pair
495,2026-07-14 21:00:00+00:00,2.5964,2.6025,2.5484,2.5583,66035.60,LIT/USDC
496,2026-07-14 22:00:00+00:00,2.5583,2.6308,2.5492,2.6128,67579.14,LIT/USDC
497,2026-07-14 23:00:00+00:00,2.6128,2.6168,2.5803,2.5912,23054.16,LIT/USDC
498,2026-07-15 00:00:00+00:00,2.5912,2.6394,2.5890,2.6195,94884.07,LIT/USDC
499,2026-07-15 01:00:00+00:00,2.6195,2.6260,2.5962,2.6033,24547.58,LIT/USDC


In [ ]:
#|eval: false
# Offline test: candles_to_df shapes raw candles correctly and removes duplicates
sample = [{'t': 1700000000000, 'o': 1.0, 'h': 2.0, 'l': 0.5, 'c': 1.5, 'v': 10.0, 'i': 1},
          {'t': 1700003600000, 'o': 1.5, 'h': 2.5, 'l': 1.0, 'c': 2.0, 'v': 20.0, 'i': 2},
          {'t': 1700003600000, 'o': 1.5, 'h': 2.5, 'l': 1.0, 'c': 2.0, 'v': 20.0, 'i': 2}]  # duplicate
df_sample = candles_to_df(sample, 'TEST/USDC')
assert len(df_sample) == 2
assert list(df_sample.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
assert str(df_sample['datetime'].dt.tz) == 'UTC'
df_sample

,datetime,open,high,low,close,volume,pair
0,2023-11-14 22:13:20+00:00,1.0,2.0,0.5,1.5,10.0,TEST/USDC
1,2023-11-14 23:13:20+00:00,1.5,2.5,1.0,2.0,20.0,TEST/USDC


In [0]:
#| echo: false
#| output: asis
show_doc(save_file)

---

[source](https://github.com/silvaac/token_data/blob/main/token_data/coinbase.py#L175){target="_blank" style="float:right; font-size:smaller"}

### save_file

```python
def save_file(
    df, folder_path, file_name, type:str='parquet'
):
```

*Save a pandas DataFrame to a file in either CSV or Parquet format.*

Args:
    df (pandas.DataFrame): The DataFrame to save
    folder_path (str): Directory path where the file will be saved
    file_name (str): Name of the file without extension
    type (str, optional): File format - either "csv" or "parquet". Defaults to "parquet"

In [0]:
#| echo: false
#| output: asis
show_doc(file_name_to_symbol)

---

### file_name_to_symbol

```python
def file_name_to_symbol(
    file_name
):
```

*Convert a file name back into a symbol.*

Example: 'LIT-USDC_1h.parquet' -> 'LIT/USDC'

In [0]:
#| echo: false
#| output: asis
show_doc(symbol_to_file_name)

---

### symbol_to_file_name

```python
def symbol_to_file_name(
    symbol, timeframe:str='1h'
):
```

*Convert a symbol and timeframe into a file name (without extension).*

Example: ('LIT/USDC', '1h') -> 'LIT-USDC_1h'

In [ ]:
#|eval: false
# Offline test: save_file round-trip and file-name helpers
import tempfile
tmp_dir = tempfile.mkdtemp()
save_file(df_sample, tmp_dir, 'TEST-USDC_1h', type='parquet')
df_back = pd.read_parquet(f"{tmp_dir}/TEST-USDC_1h.parquet")
assert df_back.shape == df_sample.shape
assert list(df_back.columns) == list(df_sample.columns)
assert symbol_to_file_name('LIT/USDC', '1h') == 'LIT-USDC_1h'
assert file_name_to_symbol('LIT-USDC_1h.parquet') == 'LIT/USDC'
print('save_file round-trip OK')

save_file round-trip OK


In [0]:
#| echo: false
#| output: asis
show_doc(lighter_to_file)

---

### lighter_to_file

```python
def lighter_to_file(
    folder_path:str='../data/lighter', token_list:list=['LIT/USDC'], type:str='parquet', timeframe:str='1h',
    refresh_hours:int=24, first_date:str='2024-01-01T00:00:00Z', all_tokens:bool=True, pause:int=1,
    base_url:str='https://mainnet.zklighter.elliot.ai', explorer_url:str='https://explorer.elliot.ai',
    verbose:bool=False
):
```

*Downloads and maintains historical Lighter candle data, saving one file per pair.*

Args:
    folder_path (str): Path where pair data files will be stored.
        Defaults to "../data/lighter"
    token_list (list): List of symbols to process. Defaults to ['LIT/USDC']
    type (str): File format to save data - either "csv" or "parquet". Defaults to "parquet"
    timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d"). Defaults to "1h"
    refresh_hours (int): Hours of the most recent data to re-download when updating.
        Defaults to 24
    first_date (str): ISO 8601 start date used for the initial full-history download.
        Defaults to '2024-01-01T00:00:00Z'
    all_tokens (bool): If True, includes any additional pairs found in the folder path.
        Defaults to True
    pause (int): Seconds to wait between pairs. Defaults to 1
    base_url (str): Lighter API base URL
    explorer_url (str): Lighter explorer API base URL for market lookup
    verbose (bool): If True, prints download progress. Defaults to False

The function:
- Creates the folder_path if it doesn't exist
- Date/Time is UTC
- For each pair, checks if a data file exists:
    - If exists: Loads the file and appends new data, refreshing the last `refresh_hours`
    - If not exists: Downloads full history starting from `first_date`
- Saves data in the specified format, handling duplicates and sorting by datetime

#### Example / tests

In [ ]:
#|eval: false
# Live test: download LIT/USDC into a temporary folder, then re-run incrementally
import tempfile
lighter_dir = tempfile.mkdtemp()
recent = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=3)).strftime('%Y-%m-%dT%H:%M:%SZ')
lighter_to_file(folder_path=lighter_dir, token_list=['LIT/USDC'], type='parquet',
                timeframe='1h', first_date=recent, pause=0)
assert os.path.exists(f"{lighter_dir}/LIT-USDC_1h.parquet")
df1 = pd.read_parquet(f"{lighter_dir}/LIT-USDC_1h.parquet")
n1 = len(df1)
assert n1 > 0
# Incremental re-run must not shrink the file and must not create duplicates
lighter_to_file(folder_path=lighter_dir, token_list=['LIT/USDC'], type='parquet',
                timeframe='1h', first_date=recent, pause=0)
df2 = pd.read_parquet(f"{lighter_dir}/LIT-USDC_1h.parquet")
assert len(df2) >= n1
assert not df2['datetime'].duplicated().any()
print(f"First run: {n1} rows, second run: {len(df2)} rows")

Processing LIT/USDC


Processing LIT/USDC


First run: 500 rows, second run: 500 rows


In [19]:
#|eval: false
# Download / update a set of spot pairs into the exchange-named data folder
lighter_to_file(folder_path="../data/lighter",
               token_list=['LIT/USDC'],
               type="parquet", timeframe='1h')

Processing LIT/USDC
